---
# <div align="center"><font color='red'>  </font></div>
# <div align="center"><font color='red'> COSC 2779/2972 | Deep Learning  </font></div>
## <div align="center"> <font color='red'> Week 4 Lecture Example: **Convolutional Neural Networks**</font></div>
---


# Setup Notebook

In [1]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split, TensorDataset

from sklearn.metrics import roc_auc_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
torch.manual_seed(42)


ModuleNotFoundError: No module named 'numpy'

In [ ]:
from itertools import cycle

def plotter(history_hold, metric='loss', ylim=[0.0, 1.0]):
    cycol = cycle('bgrcmk')
    for name, hist in history_hold.items():
        y_train = hist[metric]
        y_val = hist['val_' + metric]
        x = np.arange(len(y_val))
        c = next(cycol)
        plt.plot(x, y_train, c + '-', label=name + '_train')
        plt.plot(x, y_val, c + '--', label=name + '_val')
    plt.legend()
    plt.xlim([1, max(plt.xlim())])
    plt.ylim(ylim)
    plt.xlabel('Epoch')
    plt.ylabel(metric)
    plt.grid(True)


In [ ]:
m_histories = {}


# Dataset

For this Exercise you will use the [FashionMNIST](https://github.com/zalandoresearch/fashion-mnist) dataset. FashionMNIST contains 70,000 grayscale images in 10 categories. The images show individual articles of clothing at low resolution (28 by 28 pixels).

In [ ]:
from torchvision import datasets, transforms

# Load raw uint8 arrays first (mirrors keras.datasets.fashion_mnist.load_data)
raw_train = datasets.FashionMNIST(root='./data', train=True,  download=True)
raw_test  = datasets.FashionMNIST(root='./data', train=False, download=True)

train_images = raw_train.data.numpy()   # (60000, 28, 28), uint8
train_labels = raw_train.targets.numpy()
test_images  = raw_test.data.numpy()    # (10000, 28, 28), uint8
test_labels  = raw_test.targets.numpy()

print(train_images.shape, train_labels.shape, test_images.shape, test_labels.shape)


Each image is mapped to a single label. Since the class names are not included with the dataset, store them here to use later when plotting the images:

In [ ]:
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']


In [ ]:
i = 1
plt.figure(figsize=(12, 12))
for image, label in zip(train_images, train_labels):
    if i < 17:
        plt.subplot(4, 4, i)
        plt.imshow(image, cmap=plt.cm.binary)
        plt.title(class_names[label])
        plt.xticks([])
        plt.yticks([])
    i = i + 1

plt.show()


# Traditional CV features


Histogram of gradient (HOG) features

In [ ]:
import matplotlib.pyplot as plt

from skimage.feature import hog
from skimage import data, exposure


image = data.astronaut()

fd, hog_image = hog(image, orientations=8, pixels_per_cell=(16, 16),
                    cells_per_block=(1, 1), visualize=True, channel_axis=-1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4), sharex=True, sharey=True)

ax1.axis('off')
ax1.imshow(image, cmap=plt.cm.gray)
ax1.set_title('Input image')

# Rescale histogram for better display
hog_image_rescaled = exposure.rescale_intensity(hog_image, in_range=(0, 10))

ax2.axis('off')
ax2.imshow(hog_image_rescaled, cmap=plt.cm.gray)
ax2.set_title('Histogram of Oriented Gradients')
plt.show()


Gabor filter banks

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy import ndimage as ndi

from skimage import data
from skimage.util import img_as_float
from skimage.filters import gabor_kernel


def compute_feats(image, kernels):
    feats = np.zeros((len(kernels), 2), dtype=np.double)
    for k, kernel in enumerate(kernels):
        filtered = ndi.convolve(image, kernel, mode='wrap')
        feats[k, 0] = filtered.mean()
        feats[k, 1] = filtered.var()
    return feats


def match(feats, ref_feats):
    min_error = np.inf
    min_i = None
    for i in range(ref_feats.shape[0]):
        error = np.sum((feats - ref_feats[i, :]) ** 2)
        if error < min_error:
            min_error = error
            min_i = i
    return min_i


# prepare filter bank kernels
kernels = []
for theta in range(4):
    theta = theta / 4. * np.pi
    for sigma in (1, 3):
        for frequency in (0.05, 0.25):
            kernel = np.real(gabor_kernel(frequency, theta=theta,
                                          sigma_x=sigma, sigma_y=sigma))
            kernels.append(kernel)


shrink = (slice(0, None, 3), slice(0, None, 3))
brick = img_as_float(data.brick())[shrink]
grass = img_as_float(data.grass())[shrink]
gravel = img_as_float(data.gravel())[shrink]
image_names = ('brick', 'grass', 'gravel')
images = (brick, grass, gravel)

# prepare reference features
ref_feats = np.zeros((3, len(kernels), 2), dtype=np.double)
ref_feats[0, :, :] = compute_feats(brick, kernels)
ref_feats[1, :, :] = compute_feats(grass, kernels)
ref_feats[2, :, :] = compute_feats(gravel, kernels)

print('Rotated images matched against references using Gabor filter banks:')

print('original: brick, rotated: 30deg, match result: ', end='')
feats = compute_feats(ndi.rotate(brick, angle=190, reshape=False), kernels)
print(image_names[match(feats, ref_feats)])

print('original: brick, rotated: 70deg, match result: ', end='')
feats = compute_feats(ndi.rotate(brick, angle=70, reshape=False), kernels)
print(image_names[match(feats, ref_feats)])

print('original: grass, rotated: 145deg, match result: ', end='')
feats = compute_feats(ndi.rotate(grass, angle=145, reshape=False), kernels)
print(image_names[match(feats, ref_feats)])


def power(image, kernel):
    # Normalize images for better comparison.
    image = (image - image.mean()) / image.std()
    return np.sqrt(ndi.convolve(image, np.real(kernel), mode='wrap') ** 2 +
                   ndi.convolve(image, np.imag(kernel), mode='wrap') ** 2)

# Plot a selection of the filter bank kernels and their responses.
results = []
kernel_params = []
for theta in (0, 1):
    theta = theta / 4. * np.pi
    for frequency in (0.1, 0.4):
        kernel = gabor_kernel(frequency, theta=theta)
        params = 'theta=%d,\nfrequency=%.2f' % (theta * 180 / np.pi, frequency)
        kernel_params.append(params)
        results.append((kernel, [power(img, kernel) for img in images]))

fig, axes = plt.subplots(nrows=5, ncols=4, figsize=(5, 6))
plt.gray()

fig.suptitle('Image responses for Gabor filter kernels', fontsize=12)

axes[0][0].axis('off')

for label, img, ax in zip(image_names, images, axes[0][1:]):
    ax.imshow(img)
    ax.set_title(label, fontsize=9)
    ax.axis('off')

for label, (kernel, powers), ax_row in zip(kernel_params, results, axes[1:]):
    ax = ax_row[0]
    ax.imshow(np.real(kernel))
    ax.set_ylabel(label, fontsize=7)
    ax.set_xticks([])
    ax.set_yticks([])

    vmin = np.min(powers)
    vmax = np.max(powers)
    for patch, ax in zip(powers, ax_row[1:]):
        ax.imshow(patch, vmin=vmin, vmax=vmax)
        ax.axis('off')

plt.show()


# Data Preperation

Scale pixel values to [0, 1] before feeding them to the neural network model. It's important that the training set and the test set be preprocessed in the same way:

In [ ]:
train_images = train_images.astype('float32') / 255.0
test_images  = test_images.astype('float32')  / 255.0


## Build the datasets and a generic training loop

We wrap the numpy arrays in PyTorch `TensorDataset`s and do a 70/30 train/val split. The `train_and_track` helper trains a model and returns a history dict compatible with our `plotter`.

In [ ]:
# Add channel dim: (N, 1, 28, 28)
X_train_t = torch.from_numpy(train_images).unsqueeze(1)
y_train_t = torch.from_numpy(train_labels).long()
X_test_t  = torch.from_numpy(test_images).unsqueeze(1)
y_test_t  = torch.from_numpy(test_labels).long()

full_train = TensorDataset(X_train_t, y_train_t)
test_ds    = TensorDataset(X_test_t,  y_test_t)

n_total = len(full_train)
n_val = int(0.3 * n_total)
g = torch.Generator().manual_seed(42)
train_ds, val_ds = random_split(full_train, [n_total - n_val, n_val], generator=g)

print(f'Train: {len(train_ds)}  Val: {len(val_ds)}  Test: {len(test_ds)}')


In [ ]:
def train_and_track(model, epochs=25, batch_size=64, lr=1e-3, verbose=False):
    """Train `model` and return a history dict: loss / accuracy / val_loss / val_accuracy."""
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=256)

    hist = {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': []}
    for ep in range(epochs):
        model.train()
        tot_loss, tot_correct, tot_n = 0.0, 0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            tot_loss += loss.item() * len(xb)
            tot_correct += (out.argmax(1) == yb).sum().item()
            tot_n += len(xb)
        hist['loss'].append(tot_loss / tot_n)
        hist['accuracy'].append(tot_correct / tot_n)

        model.eval()
        vl, vc, vn = 0.0, 0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                out = model(xb)
                vl += criterion(out, yb).item() * len(xb)
                vc += (out.argmax(1) == yb).sum().item()
                vn += len(xb)
        hist['val_loss'].append(vl / vn)
        hist['val_accuracy'].append(vc / vn)

        if verbose:
            print(f'ep {ep+1:2d}  loss {hist["loss"][-1]:.3f}  acc {hist["accuracy"][-1]:.3f}  '
                  f'val_loss {hist["val_loss"][-1]:.3f}  val_acc {hist["val_accuracy"][-1]:.3f}')
    return hist


def evaluate(model, dataset):
    model.eval()
    loader = DataLoader(dataset, batch_size=256)
    criterion = nn.CrossEntropyLoss()
    tot_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            tot_loss += criterion(out, yb).item() * len(xb)
            correct += (out.argmax(1) == yb).sum().item()
            total += len(xb)
    return tot_loss / total, correct / total


def n_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# Models

Lets start with a simple model. An MLP with one hidden layer (128 neurons).

## MLP

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28 * 28, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = x.view(x.size(0), -1)          # Flatten
        x = F.relu(self.fc1(x))
        return self.fc2(x)                 # logits (CrossEntropyLoss applies log-softmax)

model1 = MLP()
print(model1)
print('Trainable parameters:', n_params(model1))


In [ ]:
m_histories['simple1H'] = train_and_track(model1, epochs=25)


In [ ]:
plotter(m_histories, ylim=[0.0, 1.1], metric='loss')


In [ ]:
train_loss, train_acc = evaluate(model1, train_ds)
test_loss, test_acc = evaluate(model1, test_ds)
print('\nTrain accuracy:', train_acc)
print('\nTest accuracy:', test_acc)


**What is your judgment of the model?**

The loss curve reveals that the model is over-fitting. Why is the model overfitting given that it is a very simple network? Should we reduce the number of hidden neurons? Try this your self.

Note: *What happens if we use the accuracy (or other metric) to determine over/under fitting?* See below:

> 🚨 **Course policy:** Peeking at the test set during the model development phase is a cardinal sin in this course. Doing so in your assignments will result in a **failing grade**. Train and tune on train/validation only; evaluate on test **exactly once** at the very end.

In [ ]:
train_loss, train_acc = evaluate(model1, train_ds)
val_loss, val_acc = evaluate(model1, val_ds)
print('\nTrain accuracy:', train_acc)
print('\nVal accuracy:', val_acc)

In [ ]:
plotter(m_histories, ylim=[0.0, 1.1], metric='accuracy')


Lets examine the model.

In [ ]:
for name, p in model1.named_parameters():
    print(f'{name:20s} {tuple(p.shape)}  {p.numel()} params')
print('Total trainable parameters:', n_params(model1))


Despite having few layers, the model has over 100,000 trainable parameters.

In [ ]:
n_dense_layer1_weights = 128 * 784 + 128
n_dense_layer2_weights = 10 * 128 + 10
print(n_dense_layer1_weights, n_dense_layer2_weights)


**How can we reduce the number of parameters?**

## 1DConv

In [ ]:
class Conv1DModel(nn.Module):
    def __init__(self):
        super().__init__()
        # PyTorch Conv1d expects (N, C, L). We reshape (N, 28, 28) -> (N, 1, 784).
        self.conv1 = nn.Conv1d(in_channels=1,   out_channels=128, kernel_size=3)
        self.conv2 = nn.Conv1d(in_channels=128, out_channels=10,  kernel_size=3)

    def forward(self, x):
        x = x.view(x.size(0), 1, -1)       # (N, 1, 784)
        x = F.relu(self.conv1(x))          # (N, 128, 782)
        x = F.relu(self.conv2(x))          # (N,  10, 780)
        return x.mean(dim=-1)              # GlobalAveragePooling1D -> (N, 10)

model2 = Conv1DModel()
print(model2)
print('Trainable parameters:', n_params(model2))


**Why is this not a good option?**

## Simple 2DConv

In [ ]:
class Simple2DConv(nn.Module):
    def __init__(self):
        super().__init__()
        # first layer
        self.conv1 = nn.Conv2d(1,  32, kernel_size=3, padding=1)   # padding='same' for 3x3
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        # second layer
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool2 = nn.AvgPool2d(kernel_size=2, stride=2)

        # MLP classifier
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool1(F.relu(self.conv1(x)))   # (N, 32, 14, 14)
        x = self.pool2(F.relu(self.conv2(x)))   # (N, 64,  7,  7)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)                      # logits

model3 = Simple2DConv()
print(model3)
print('Trainable parameters:', n_params(model3))


This model also has ~421k parameters. However, if you look closely the convolutional layers have considerably fewer parameters. Most of the parameters (over 400,000) are in the MLP head.

In [ ]:
m_histories['simple2DConv'] = train_and_track(model3, epochs=25)
plotter(m_histories, ylim=[0.0, 1.1], metric='loss')

train_loss, train_acc = evaluate(model3, train_ds)
val_loss, val_acc = evaluate(model3, val_ds)
print('\nTrain accuracy:', train_acc)
print('\nVal accuracy:', val_acc)


**What is your judgment on the trained model?**

It looks like it is still overfitting. What should be the next step? Should we randomly try out different combinations?

## Regularised Model

In [ ]:
class RegularisedCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # first block: Conv -> BN -> ReLU -> MaxPool -> Dropout
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1   = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.drop1 = nn.Dropout(0.25)

        # second block: Conv -> BN -> ReLU -> AvgPool -> Dropout
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2   = nn.BatchNorm2d(64)
        self.pool2 = nn.AvgPool2d(2, 2)
        self.drop2 = nn.Dropout(0.3)

        # MLP classifier
        self.fc1   = nn.Linear(64 * 7 * 7, 128)
        self.drop3 = nn.Dropout(0.3)
        self.fc2   = nn.Linear(128, 10)

    def forward(self, x):
        x = self.drop1(self.pool1(F.relu(self.bn1(self.conv1(x)))))
        x = self.drop2(self.pool2(F.relu(self.bn2(self.conv2(x)))))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.drop3(x)
        return self.fc2(x)

model4 = RegularisedCNN()
print(model4)
print('Trainable parameters:', n_params(model4))


In [ ]:
m_histories['Regularised'] = train_and_track(model4, epochs=25)
plotter(m_histories, ylim=[0.0, 1.1], metric='loss')

train_loss, train_acc = evaluate(model4, train_ds)
val_loss, val_acc = evaluate(model4, val_ds)
print('\nTrain accuracy:', train_acc)
print('\nVal accuracy:', val_acc)


**Again, what is your judgment of the model?**

Now we cannot see any overfitting. However, we might be having some underfitting. What should we do?

Let's assume we are satisfied with the final loss the model has achieved and that there is no underfitting. What additional evaluations should we do before we decide on deploying the model in the wild?

# Evaluate model

We can now evaluate the model with the held-out test set.

## detailed statistics

In [ ]:
from sklearn.metrics import classification_report

model4.eval()
with torch.no_grad():
    logits = model4(X_test_t.to(device))
    predicted_classes = logits.argmax(dim=1).cpu().numpy()

print(classification_report(test_labels, predicted_classes, target_names=class_names))


## Error analysis

In [ ]:
from sklearn.metrics import confusion_matrix

cf_matrix = confusion_matrix(test_labels, predicted_classes)


In [ ]:
import seaborn as sns
plt.figure(figsize=(10, 10))
ax = sns.heatmap(cf_matrix, annot=True, cmap='Blues', fmt='d')

ax.set_title('Confusion Matrix\n')
ax.set_xlabel('\nPredicted Values')
ax.set_ylabel('Actual Values ')

ax.xaxis.set_ticklabels(class_names)
ax.yaxis.set_ticklabels(class_names)
plt.yticks(rotation=0)
plt.xticks(rotation=90)
plt.show()


## View incorrectly classified test data

In [ ]:
incorrect = np.nonzero(predicted_classes != test_labels)[0]

# Display the first 8 incorrectly classified images from the test data set
plt.figure(figsize=(15, 8))
for j, idx in enumerate(incorrect[0:8]):
    plt.subplot(2, 4, j + 1)
    plt.xticks([])
    plt.yticks([])
    plt.imshow(test_images[idx].reshape(28, 28), cmap=plt.cm.binary)
    plt.title('Predicted: {}'.format(class_names[predicted_classes[idx]]))
    plt.xlabel('Actual: {}'.format(class_names[test_labels[idx]]))
plt.show()


## Using Explainable techniques

We will cover explainable techniques for deep learning later in the course. For now let's plot some feature maps from a convolution layer and see if they reveal useful information.

Inspired by this blogpost: https://towardsdatascience.com/visualizing-intermediate-activation-in-convolutional-neural-networks-with-keras-260b36d60d0.

In PyTorch, the cleanest way to grab an intermediate activation is with a **forward hook** — a small callback we register on a chosen module. When we run a forward pass, PyTorch stores that module's output for us.

In [ ]:
# Attach a forward hook to the first conv layer of the trained regularised CNN
activations = {}

def save_activation(module, inp, out):
    activations['conv1'] = out.detach().cpu()

handle = model4.conv1.register_forward_hook(save_activation)

# Choose an image (0 to 59999) from the training set and add the batch dim
image_idx = 0
image = X_train_t[image_idx:image_idx + 1].to(device)

model4.eval()
with torch.no_grad():
    _ = model4(image)

handle.remove()  # tidy up so the hook doesn't fire on future forward passes

feature_maps = activations['conv1']    # (1, 32, 28, 28)
print('Feature map tensor shape:', feature_maps.shape)

plt.figure(figsize=(15, 8))
for i in range(32):
    plt.subplot(4, 8, i + 1)
    plt.xticks([])
    plt.yticks([])
    plt.imshow(feature_maps[0, i].numpy(), cmap=plt.cm.binary)
plt.show()
